# GPGN268 - Geophysical Data Analysis
## Final Project - Martian Subsurface Analysis

**Students:** Petra Elrod, Shleby Layne, Addy Peterson
**Collaborators:**

**Date:** 1 May 2026

## Introduction / Problem Statement

**1. Data Processing:**

In [8]:
#Imports

import pandas as pd
import matplotlib.pyplot as plt

import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
# 1. Define a function to read the mars catalog file
def load_mars_catalog(filename, quality_label):
    
    cols = ['Event_ID', 'Timestamp', 'Distance', 'Quality', 'Class']
    
    df = pd.read_csv(filename, sep='\s+', names=cols, on_bad_lines='skip')
    
    df['Quality'] = quality_label
    return df

# 2. Load the data
df_a = load_mars_catalog('high_quality_events/quality_A_events.txt', 'A')
df_b = load_mars_catalog('high_quality_events/quality_B_events.txt', 'B')

# 3. Combine into one master DataFrame for the group
df_total = pd.concat([df_a, df_b], ignore_index=True)

# 4. Convert Timestamp to actual datetime objects for math later
df_total['Timestamp'] = pd.to_datetime(df_total['Timestamp'], errors='coerce')

# 5. Quick look at the "Evidence"
print(f"Successfully loaded {len(df_total)} events.")

#Save as a new data file
df_total.to_csv('marsquakes_data_frame.csv', index=False) #We can rename this file to

df_total.head(5)

Successfully loaded 185 events.


<>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/60/trjb6m6100qft2g2gpz2xqh00000gn/T/ipykernel_44534/3850015462.py:6: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  df = pd.read_csv(filename, sep='\s+', names=cols, on_bad_lines='skip')


,Event_ID,Timestamp,Distance,Quality,Class
0,S1222a,2022-05-04 23:23:07.856945+00:00,37.01,A,BROADBAND
1,S1133c,2022-02-03 08:04:38.145711+00:00,30.19,A,BROADBAND
2,S1102a,2022-01-02 04:27:10.093677+00:00,73.31,A,BROADBAND
3,S1094b,2021-12-24 22:38:02.749173+00:00,59.65,A,BROADBAND
4,S1048d,2021-11-07 22:00:15.254098+00:00,30.19,A,LOW_FREQUENCY


In [10]:
#Parse the xml file to get the vp and vs waves

# 1. Load the XML data
xml_file = 'dev/events_mars_extended_multiorigin_v12_2022-07-01.xml'
tree = ET.parse(xml_file)
root = tree.getroot()

#Create a custom xml parser
def get_event_picks(event_id, root):
    """
    Finds the earliest P-wave and the first S-wave arriving after it.
    Bypasses XML namespaces using wildcards.
    """
    for event in root.findall('.//{*}event'):
        public_id = event.get('publicID', '')
        # Extract all text in the event block to find a specific ID
        all_text = "".join([t.text for t in event.findall('.//{*}text') if t.text])
        
        if event_id.lower() in public_id.lower() or event_id.lower() in all_text.lower():
            # Find the Origin Time (t0)
            origin_node = event.find('.//{*}origin/{*}time/{*}value')
            t0 = origin_node.text if origin_node is not None else None
            
            p_arrivals, s_arrivals = [], []
            
            # Search through all 'picks' for P and S phase hints
            for pick in event.findall('.//{*}pick'):
                phase = pick.find('{*}phaseHint')
                time_node = pick.find('{*}time/{*}value')
                
                if phase is not None and time_node is not None:
                    p_text = phase.text.upper()
                    t_val = time_node.text
                    if 'P' in p_text: p_arrivals.append(t_val)
                    elif 'S' in p_text: s_arrivals.append(t_val)
            
            # Physics Constraint: Earliest P, and the first S that happens AFTER P
            if p_arrivals and s_arrivals:
                tp = min(p_arrivals)
                valid_s = [s for s in s_arrivals if s > tp]
                if valid_s:
                    ts = min(valid_s)
                    return t0, tp, ts
                    
    return None, None, None

In [11]:
# 1. Create a new dataframe from total data
# Grab inmportant info we already have to initiate a new dataframe
seismic_df = df_total[['Event_ID','Distance']].copy()

# 2. Add the empty columns 
seismic_df['t0'] = None
seismic_df['tp'] = None
seismic_df['ts'] = None
seismic_df['Vp_Vs_Ratio'] = np.nan

print(f"Starting calculation for {len(seismic_df)} events...")

# 3. Run the Robust Calculation Loop
for index, row in seismic_df.iterrows():
    t0, tp, ts = get_event_picks(row['Event_ID'], root)
    
    seismic_df.at[index, 't0'] = t0
    seismic_df.at[index, 'tp'] = tp
    seismic_df.at[index, 'ts'] = ts

    # Calculate if all phases were found
    if all([t0, tp, ts]):
        try:
            # ISO format for NASA InSight data
            fmt = "%Y-%m-%dT%H:%M:%S.%fZ"
            t0_dt = datetime.strptime(t0, fmt)
            tp_dt = datetime.strptime(tp, fmt)
            ts_dt = datetime.strptime(ts, fmt)
            
            # Calculate Travel Times (seconds)
            p_travel = (tp_dt - t0_dt).total_seconds()
            s_travel = (ts_dt - t0_dt).total_seconds()
            
            # Calculate the ratio: Vp/Vs = S_travel_time / P_travel_time
            # Since distance is the same for both, Vp/Vs = (dist/tp) / (dist/ts) = ts/tp
            if p_travel > 0 and s_travel > p_travel:
                seismic_df.at[index, 'Vp_Vs_Ratio'] = s_travel / p_travel
        except Exception:
            # If the timestamp format is slightly different for an event, skip it
            continue

# Drop events that didn't have enough data for a ratio
seismic_df = seismic_df.dropna(subset=['Vp_Vs_Ratio']).copy()

seismic_df.head(30)

Starting calculation for 185 events...


,Event_ID,Distance,t0,tp,ts,Vp_Vs_Ratio
0,S1222a,37.01,2022-05-04T23:23:07.856945Z,2022-05-04T23:27:34.0Z,2022-05-04T23:27:45.331451Z,1.042577
1,S1133c,30.19,2022-02-03T08:04:38.145711Z,2022-02-03T08:08:11.7Z,2022-02-03T08:08:25.290742Z,1.063641
2,S1102a,73.31,2022-01-02T04:27:10.093677Z,2022-01-02T04:35:19.3Z,2022-01-02T04:35:28.844688Z,1.019511
3,S1094b,59.65,2021-12-24T22:38:02.749173Z,2021-12-24T22:44:48.7Z,2021-12-24T22:45:08.063325Z,1.047699
4,S1048d,30.19,2021-11-07T22:00:15.254098Z,2021-11-07T22:03:42.7Z,2021-11-07T22:04:04.40886Z,1.104648
5,S1022a,30.73,2021-10-11T23:14:29.105382Z,2021-10-11T23:18:25.025302Z,2021-10-11T23:21:23.255908Z,1.755471
6,S1015f,27.49,2021-10-04T04:52:29.248537Z,2021-10-04T04:56:00.185082Z,2021-10-04T04:58:40.218707Z,1.758681
7,S1000a,128.29,2021-09-18T17:46:20.639751Z,2021-09-18T17:59:01.470476Z,2021-09-18T18:13:28.0Z,2.138926
8,S0976a,146.26,2021-08-25T03:32:20.629953Z,2021-08-25T03:48:58.4Z,2021-08-25T04:03:07.778327Z,1.851277
9,S0864a,28.75,2021-05-02T00:57:35.34519Z,2021-05-02T01:00:56.5Z,2021-05-02T01:01:05.534422Z,1.044913


In [12]:
#Cleaning and Editing

#Add Kilometers Column
# 1. Define the Martian conversion factor (1 degree = ~59.2 km)
km_per_degree = 59.2

# 2. Calculate Distance in KM
seismic_df['Distance_KM'] = seismic_df['Distance'] * km_per_degree

# 3. Organize the columns for the final report
# This keeps your locations, paired distances, timestamps, and ratio
order = [
    'Event_ID', 'Distance', 'Distance_KM', 
    't0', 'tp', 'ts', 
    'Vp_Vs_Ratio'
]

# Apply the new order
seismic_df = seismic_df[order]

# 4. Rename the distance columns for professional labeling
seismic_df = seismic_df.rename(columns={
    'Distance': 'Distance_Degrees'
})

#Dealing with Outliers

# Filter for physically plausible ratios only
clean_seismic_df = seismic_df[(seismic_df['Vp_Vs_Ratio'] >= 1.4) & (seismic_df['Vp_Vs_Ratio'] <= 2.4)]

# 5. Final Save to your local directory
clean_seismic_df.to_csv('final_marsquakes_data_frame.csv', index=False) #we can rename this 
clean_seismic_df.head()

,Event_ID,Distance_Degrees,Distance_KM,t0,tp,ts,Vp_Vs_Ratio
5,S1022a,30.73,1819.216,2021-10-11T23:14:29.105382Z,2021-10-11T23:18:25.025302Z,2021-10-11T23:21:23.255908Z,1.755471
6,S1015f,27.49,1627.408,2021-10-04T04:52:29.248537Z,2021-10-04T04:56:00.185082Z,2021-10-04T04:58:40.218707Z,1.758681
7,S1000a,128.29,7594.768,2021-09-18T17:46:20.639751Z,2021-09-18T17:59:01.470476Z,2021-09-18T18:13:28.0Z,2.138926
8,S0976a,146.26,8658.592,2021-08-25T03:32:20.629953Z,2021-08-25T03:48:58.4Z,2021-08-25T04:03:07.778327Z,1.851277
10,S0820a,30.19,1787.248,2021-03-18T14:51:33.869889Z,2021-03-18T14:54:39.0Z,2021-03-18T14:57:39.0Z,1.972289
